# Practical 0 — Les bases de PyTorch (correction commentée)

Ce notebook corrige le **Practical 0** pas à pas. L'objectif n'est pas d'écrire le code le plus court, mais de **comprendre** chaque brique qu'on réutilisera ensuite pour construire GPT-2 :

1. **Les tenseurs** : les « tableaux de nombres » de PyTorch.
2. **Les gradients** : comment PyTorch calcule tout seul les dérivées (c'est ce qui permet d'apprendre).
3. **Les réseaux de neurones, Dataset et DataLoader** : définir un modèle, lui donner des données, l'entraîner.
4. **Les devices** : calculer sur CPU ou GPU.

> 💡 Conseil : exécute les cellules **dans l'ordre** (Shift + Entrée) et lis les commentaires avant de regarder le résultat.

---
## Partie 1 — Les matrices (tenseurs) avec PyTorch

### Q1. Importer `torch` et afficher la version
On vérifie que la version est au moins **2.1.2**.

In [1]:
import torch

print("Version de PyTorch :", torch.__version__)

Version de PyTorch : 2.14.1+cpu


### Q2. Créer des tenseurs de différentes dimensions

Un **tenseur** est simplement un tableau de nombres, comme une liste Python, mais optimisé pour les calculs.
Le nombre de dimensions dit « combien d'indices il faut pour atteindre un nombre » :

| Dimension | Nom courant | Exemple |
|---|---|---|
| 0 | scalaire | `5` |
| 1 | vecteur | `[1, 2, 3]` |
| 2 | matrice | `[[1, 2], [3, 4]]` |
| 3+ | tenseur | un « cube » de nombres |

- `.ndim` donne le nombre de dimensions,
- `.shape` donne la taille de chaque dimension.

In [2]:
scalaire = torch.tensor(5)                       # 0 dimension
vecteur  = torch.tensor([1, 2, 3])               # 1 dimension
matrice  = torch.tensor([[1, 2], [3, 4]])        # 2 dimensions
tenseur3 = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])      # 3 dimensions

for nom, t in [("scalaire", scalaire), ("vecteur", vecteur), ("matrice", matrice), ("tenseur 3D", tenseur3)]:
    print(f"{nom:10} -> ndim = {t.ndim}, shape = {tuple(t.shape)}")

scalaire   -> ndim = 0, shape = ()
vecteur    -> ndim = 1, shape = (3,)
matrice    -> ndim = 2, shape = (2, 2)
tenseur 3D -> ndim = 3, shape = (2, 2, 2)


### Q3. Depuis un tableau NumPy : `torch.tensor` vs `torch.from_numpy`

- `torch.tensor(array)` fait une **copie** : le tenseur est indépendant du tableau NumPy.
- `torch.from_numpy(array)` **partage la mémoire** : si on modifie le tableau NumPy, le tenseur change aussi (et inversement).

On le vérifie en modifiant le tableau NumPy après coup 👇

In [3]:
import numpy as np

array3d = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

t_copie   = torch.tensor(array3d)       # copie
t_partage = torch.from_numpy(array3d)   # mémoire partagée

array3d[0, 0, 0] = 999   # on modifie le tableau NumPy

print("torch.tensor     (copie)   :", t_copie[0, 0, 0].item())    # reste 1
print("torch.from_numpy (partagé) :", t_partage[0, 0, 0].item())  # devient 999

torch.tensor     (copie)   : 1
torch.from_numpy (partagé) : 999


✅ **À retenir** : `from_numpy` est plus rapide (pas de copie) mais les deux objets sont liés. `torch.tensor` est plus « sûr ».

### Q4. Changer la forme : `reshape` vs `view`

On crée une matrice 2×3 et on la transforme en 3×2. Les **6 nombres restent les mêmes**, dans le même ordre ; seule la façon de les ranger change.

- `view` : ne copie jamais les données. Ne marche que si le tenseur est rangé de façon **contiguë** en mémoire, sinon il lève une erreur.
- `reshape` : fait comme `view` quand c'est possible, et **copie** les données sinon. Il marche donc toujours.

In [4]:
M = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
print("M (2x3) :\n", M)

print("\nM.view(3, 2) :\n", M.view(3, 2))
print("\nM.reshape(3, 2) :\n", M.reshape(3, 2))

M (2x3) :
 tensor([[1, 2, 3],
        [4, 5, 6]])

M.view(3, 2) :
 tensor([[1, 2],
        [3, 4],
        [5, 6]])

M.reshape(3, 2) :
 tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [5]:
# Exemple où view échoue : la transposée n'est plus "contiguë" en mémoire
Mt = M.T
try:
    Mt.view(6)
except RuntimeError as e:
    print("view a échoué :", str(e)[:80], "...")

print("reshape fonctionne :", Mt.reshape(6))

view a échoué : view size is not compatible with input tensor's size and stride (at least one di ...
reshape fonctionne : tensor([1, 4, 2, 5, 3, 6])


### Q5. Multiplier la matrice 2×3 par sa transposée

- `M.T` est la transposée (3×2).
- `@` est le produit matriciel.

Règle des tailles : **(2×3) @ (3×2) → (2×2)** (les dimensions « du milieu » doivent être égales et disparaissent).

In [6]:
resultat = M @ M.T
print(resultat)
print("Taille :", resultat.shape)   # torch.Size([2, 2])

tensor([[14, 32],
        [32, 77]])
Taille : torch.Size([2, 2])


---
## Partie 2 — Les gradients

On construit le plus petit « réseau » possible : **un seul neurone**, comme en régression logistique.

$$z = x_1 \cdot w_1 + b \qquad a = \sigma(z) = \frac{1}{1+e^{-z}}$$

- `x_1` est l'entrée, `y` la vraie réponse (1 = classe positive),
- `w_1` (poids) et `b` (biais) sont les **paramètres à apprendre**.

### Q1 → Q3. Créer les tenseurs, calculer `z` puis `a`

In [7]:
y   = torch.tensor([1.0])   # vraie étiquette
x_1 = torch.tensor([1.1])   # entrée
w_1 = torch.tensor([2.2])   # poids
b   = torch.tensor([0.0])   # biais

z = x_1 * w_1 + b
a = torch.sigmoid(z)        # la sigmoïde écrase z entre 0 et 1 -> une "probabilité"

print("z =", z)
print("a =", a)

z = tensor([2.4200])
a = tensor([0.9183])


### Q4. La perte (loss) : *binary cross-entropy*

La loss mesure **à quel point le modèle se trompe**. Pour une classification binaire :

$$\text{BCE}(a, y) = -\big[\,y \cdot \log(a) + (1-y)\cdot\log(1-a)\,\big]$$

Ici `y = 1`, donc le second terme disparaît et il reste simplement $-\log(a)$.
- Si `a` est proche de 1 (bonne réponse confiante) → loss proche de 0.
- Si `a` est proche de 0 (mauvaise réponse) → loss très grande.

In [8]:
import torch.nn.functional as F

loss = F.binary_cross_entropy(a, y)
print("loss PyTorch         :", loss.item())

# Vérification à la main
print("calcul à la main -log(a) :", (-torch.log(a)).item())

loss PyTorch         : 0.0851878821849823
calcul à la main -log(a) : 0.0851878821849823


### Q5 → Q6. Déclarer `w_1` et `b` comme entraînables et calculer les gradients

- `requires_grad=True` dit à PyTorch : « **retiens toutes les opérations** faites avec ce tenseur ». Il construit alors un *graphe de calcul*.
- `grad(loss, w_1)` parcourt ce graphe à l'envers pour calculer $\frac{\partial\, loss}{\partial w_1}$.

**À quoi sert `retain_graph=True` ?** Par défaut, PyTorch **efface le graphe** après un calcul de gradient pour libérer la mémoire. Comme on veut appeler `grad` **deux fois** (pour `w_1` puis pour `b`), on demande de garder le graphe au premier appel.

> 📐 Pour vérifier : avec la sigmoïde + BCE, les formules sont très simples :
> $\frac{\partial L}{\partial w_1} = (a - y)\cdot x_1$ et $\frac{\partial L}{\partial b} = (a - y)$

In [9]:
from torch.autograd import grad

# On recrée w_1 et b, cette fois "entraînables"
w_1 = torch.tensor([2.2], requires_grad=True)
b   = torch.tensor([0.0], requires_grad=True)

z = x_1 * w_1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

grad_w1 = grad(loss, w_1, retain_graph=True)   # on garde le graphe...
grad_b  = grad(loss, b,   retain_graph=True)   # ...pour pouvoir le réutiliser ici

print("dL/dw_1 =", grad_w1)
print("dL/db   =", grad_b)

# Vérification avec les formules
print("\nformule (a - y) * x_1 =", ((a - y) * x_1).item())
print("formule (a - y)       =", (a - y).item())

dL/dw_1 = (tensor([-0.0898]),)
dL/db   = (tensor([-0.0817]),)

formule (a - y) * x_1 = -0.089826300740242
formule (a - y)       = -0.08166027069091797


### Q7. Afficher `w_1.grad` et `b.grad`

`grad(...)` **renvoie** le gradient mais ne le **stocke pas** dans les tenseurs. Donc `w_1.grad` vaut encore `None`.

In [10]:
print("w_1.grad :", w_1.grad)
print("b.grad   :", b.grad)

w_1.grad : None
b.grad   : None


### Q8. Utiliser `loss.backward()`

`backward()` calcule **tous** les gradients d'un coup et les **range** dans l'attribut `.grad` de chaque paramètre. C'est la méthode qu'on utilise en pratique pendant l'entraînement.

In [11]:
loss.backward()

print("w_1.grad :", w_1.grad)   # même valeur que grad_w1
print("b.grad   :", b.grad)     # même valeur que grad_b

w_1.grad : tensor([-0.0898])
b.grad   : tensor([-0.0817])


---
## Partie 3 — Réseaux de neurones et DataLoaders

### Q1 & Q2. La classe `NeuralNetwork`

Pour créer un modèle avec PyTorch, on crée une classe qui **hérite de `torch.nn.Module`** et on écrit 2 méthodes :
- `__init__` : on **déclare** les couches,
- `forward` : on dit **comment les données traversent** les couches.

Architecture demandée :
```
entrée (num_inputs) → Linear(→30) → ReLU → Linear(→20) → ReLU → Linear(→num_outputs)
```
- `nn.Linear(a, b)` : calcule `x @ W + biais`, transforme un vecteur de taille `a` en vecteur de taille `b`.
- `nn.ReLU()` : remplace les nombres négatifs par 0 (c'est la « non-linéarité », sans elle le réseau ne serait qu'une grosse multiplication de matrices).
- `nn.Sequential` : enchaîne les couches dans l'ordre.

In [12]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()   # obligatoire : initialise la partie nn.Module

        self.layers = torch.nn.Sequential(
            # 1ère couche cachée
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            # 2ème couche cachée
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # couche de sortie
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        return self.layers(x)   # les "logits" (scores bruts, pas encore des probabilités)


model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


### Q3. Nombre total de paramètres entraînables

Une couche `Linear(a, b)` possède une matrice de poids `b × a` **plus** `b` biais, soit `a·b + b` paramètres.

Pour `NeuralNetwork(50, 3)` :
- couche 1 : 50·30 + 30 = **1530**
- couche 2 : 30·20 + 20 = **620**
- couche 3 : 20·3 + 3 = **63**
- **Total = 2213**

`p.numel()` donne le nombre d'éléments d'un tenseur.

In [13]:
nb_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Nombre de paramètres entraînables :", nb_params)

Nombre de paramètres entraînables : 2213


### Q4. Paramètres de la première couche cachée
`model.layers[0]` est la première couche du `Sequential`.

In [14]:
premiere_couche = model.layers[0]
print("Poids  :", premiere_couche.weight.shape)   # (30, 50)
print("Biais  :", premiere_couche.bias.shape)     # (30,)
print(premiere_couche.weight)

Poids  : torch.Size([30, 50])
Biais  : torch.Size([30])
Parameter containing:
tensor([[-0.0125,  0.0850,  0.0819,  ...,  0.0907,  0.0467, -0.0030],
        [ 0.0335,  0.0810, -0.0028,  ...,  0.0182,  0.0769,  0.1291],
        [ 0.1323,  0.1327,  0.0351,  ..., -0.0377, -0.0193,  0.1250],
        ...,
        [-0.0219, -0.0522, -0.0479,  ..., -0.1041, -0.0136, -0.0996],
        [-0.0015, -0.0524, -0.1031,  ..., -0.0314, -0.0412, -0.0068],
        [ 0.0770, -0.1020, -0.0061,  ..., -0.0504,  0.1086, -0.0440]],
       requires_grad=True)


### Q5 & Q6. Faire passer une entrée aléatoire, puis appliquer un softmax

- `torch.no_grad()` : on dit à PyTorch de **ne pas construire le graphe** de gradients. Utile quand on veut juste prédire (plus rapide, moins de mémoire).
- La sortie du modèle (les **logits**) peut contenir n'importe quels nombres. Le **softmax** les transforme en **probabilités** (positives, somme = 1).

In [15]:
torch.manual_seed(123)          # pour avoir toujours les mêmes nombres aléatoires
X = torch.rand((1, 50))         # 1 exemple de 50 valeurs

with torch.no_grad():
    logits = model(X)

print("Logits       :", logits)
probas = torch.softmax(logits, dim=1)   # dim=1 : on normalise sur les 3 classes
print("Probabilités :", probas)
print("Somme        :", probas.sum().item())

Logits       : 

tensor([[-0.1566,  0.1288,  0.1209]])
Probabilités : tensor([[0.2740, 0.3645, 0.3616]])
Somme        : 1.0


### Q7. Créer un `Dataset`

Un `Dataset` PyTorch est une classe qui sait :
- `__getitem__(index)` → renvoyer **un exemple** (entrée, étiquette),
- `__len__()` → dire **combien** d'exemples il y a.

> ⚠️ Remarque : dans l'énoncé `y_test = [0, 0]`, mais le point `[2.6, -1.6]` ressemble clairement aux exemples de classe 1 du train (`[2.3, -1.1]`, `[2.7, -1.5]`). C'est sûrement une coquille, on utilise donc `y_test = [0, 1]`.

In [16]:
from torch.utils.data import Dataset

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5],
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return len(self.labels)


train_ds = ToyDataset(X_train, y_train)
test_ds  = ToyDataset(X_test, y_test)

print("Taille du train :", len(train_ds))
print("Exemple n°0     :", train_ds[0])

Taille du train : 5
Exemple n°0     : (tensor([-1.2000,  3.1000]), tensor(0))


### Q8. Créer les `DataLoader`

Le `DataLoader` découpe le dataset en **mini-lots (batches)** :
- `batch_size=2` : 2 exemples par lot,
- `shuffle=True` : mélange l'ordre à chaque époque (seulement pour le **train**, pour que le modèle n'apprenne pas l'ordre ; inutile pour le test).

In [17]:
from torch.utils.data import DataLoader

torch.manual_seed(123)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=2, shuffle=False)

for i, (x, y) in enumerate(train_loader):
    print(f"Batch {i} : x = {x.tolist()}, y = {y.tolist()}")

Batch 0 : x = [[2.299999952316284, -1.100000023841858], [-0.8999999761581421, 2.9000000953674316]], y = [1, 0]
Batch 1 : x = [[-1.2000000476837158, 3.0999999046325684], [-0.5, 2.5999999046325684]], y = [0, 0]
Batch 2 : x = [[2.700000047683716, -1.5]], y = [1]


👀 **Ce qu'on remarque** : on a 5 exemples et des lots de 2 → 2 + 2 + **1**. Le **dernier lot ne contient qu'un seul exemple**. Un lot trop petit donne une estimation du gradient peu fiable et peut perturber l'entraînement.

### Q9. Avoir des lots tous de la même taille
On ajoute `drop_last=True` : le dernier lot incomplet est **ignoré**.

In [18]:
torch.manual_seed(123)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, drop_last=True)

for i, (x, y) in enumerate(train_loader):
    print(f"Batch {i} : x = {x.tolist()}, y = {y.tolist()}")

Batch 0 : x = [[2.299999952316284, -1.100000023841858], [-0.8999999761581421, 2.9000000953674316]], y = [1, 0]
Batch 1 : x = [[-1.2000000476837158, 3.0999999046325684], [-0.5, 2.5999999046325684]], y = [0, 0]


### Q10. Entraîner le modèle sur 3 époques

Une **époque** = un passage complet sur toutes les données. Pour **chaque lot**, on répète toujours les **5 mêmes étapes** (à connaître par cœur, c'est la boucle d'entraînement de tout modèle PyTorch, GPT compris) :

1. `logits = model(x)` → prédiction
2. `loss = F.cross_entropy(logits, y)` → mesure de l'erreur
3. `optimizer.zero_grad()` → **remet les gradients à zéro** (sinon PyTorch les additionne d'un lot à l'autre)
4. `loss.backward()` → calcule les gradients
5. `optimizer.step()` → met à jour les poids : `w ← w - lr × gradient`

> `F.cross_entropy` applique le softmax **lui-même** : on lui donne directement les logits.

In [19]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)   # lr = "learning rate" = taille du pas

num_epochs = 3
for epoch in range(num_epochs):
    model.train()                       # mode entraînement
    for batch_idx, (x, y) in enumerate(train_loader):
        logits = model(x)                       # 1. prédiction
        loss = F.cross_entropy(logits, y)       # 2. erreur

        optimizer.zero_grad()                   # 3. remise à zéro
        loss.backward()                         # 4. gradients
        optimizer.step()                        # 5. mise à jour

        print(f"Époque {epoch+1}/{num_epochs} | Batch {batch_idx} | Loss : {loss.item():.4f}")

Époque 1/3 | Batch 0 | Loss : 0.7487
Époque 1/3 | Batch 1 | Loss : 0.6450
Époque 2/3 | Batch 0 | Loss : 0.4423
Époque 2/3 | Batch 1 | Loss : 0.1256
Époque 3/3 | Batch 0 | Loss : 0.0269
Époque 3/3 | Batch 1 | Loss : 0.0043


### Q11. Prédictions sur le jeu d'entraînement

- `model.eval()` : passe en mode évaluation (important quand il y a du dropout, qu'on verra au Practical 1).
- `torch.argmax(probas, dim=1)` : pour chaque exemple, l'indice de la classe **la plus probable**.

In [20]:
model.eval()
with torch.no_grad():
    logits = model(X_train)

probas = torch.softmax(logits, dim=1)
predictions = torch.argmax(probas, dim=1)

print("Probabilités :\n", probas)
print("Prédictions  :", predictions)
print("Vraies classes :", y_train)

Probabilités :
 tensor([[9.9911e-01, 8.9419e-04],
        [9.9815e-01, 1.8458e-03],
        [9.9491e-01, 5.0852e-03],
        [4.9127e-02, 9.5087e-01],
        [3.0714e-02, 9.6929e-01]])
Prédictions  : tensor([0, 0, 0, 1, 1])
Vraies classes : tensor([0, 0, 0, 1, 1])


### Q12. Précision (accuracy) sur le train et le test

Accuracy = nombre de bonnes prédictions / nombre total d'exemples.

In [21]:
def compute_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    for x, y in dataloader:
        with torch.no_grad():
            logits = model(x)
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == y).sum().item()   # nombre de bonnes réponses dans le lot
        total += len(y)
    return correct / total

# On recrée un loader de train SANS drop_last pour évaluer sur tous les exemples
train_eval_loader = DataLoader(train_ds, batch_size=2, shuffle=False)

print("Accuracy train :", compute_accuracy(model, train_eval_loader))
print("Accuracy test  :", compute_accuracy(model, test_loader))

Accuracy train : 1.0
Accuracy test  : 1.0


### Q13 & Q14. Sauvegarder puis recharger le modèle

- On ne sauvegarde pas tout l'objet, seulement le **`state_dict`** : un dictionnaire `{nom de couche : poids}`.
- Pour recharger, on recrée un modèle **avec la même architecture**, puis on y charge les poids.

In [22]:
torch.save(model.state_dict(), "model.pth")
print("Modèle sauvegardé dans model.pth")

nouveau_model = NeuralNetwork(2, 2)                         # même architecture, poids aléatoires
nouveau_model.load_state_dict(torch.load("model.pth"))      # on charge les poids appris

print("Accuracy train du modèle rechargé :", compute_accuracy(nouveau_model, train_eval_loader))

Modèle sauvegardé dans model.pth
Accuracy train du modèle rechargé : 1.0


---
## Partie 4 — Les devices (CPU / GPU)

### Q1. Quel matériel est disponible ?
- `cuda` = carte graphique **NVIDIA**,
- `mps` = puce **Apple** (Mac M1/M2/...),
- `cpu` = le processeur, toujours disponible.

Règle d'or : **le modèle et les données doivent être sur le même device**, sinon PyTorch lève une erreur.

In [23]:
print("CUDA disponible :", torch.cuda.is_available())
print("MPS disponible  :", torch.backends.mps.is_available())
print("CPU disponible  :", torch.cpu.is_available())

# On choisit automatiquement le meilleur device disponible
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("\nDevice utilisé :", device)

CUDA disponible : False
MPS disponible  : False
CPU disponible  : True

Device utilisé : cpu


### Q2. Entraîner sur ce device

Seuls 2 changements par rapport à la Q10 :
- `model.to(device)` une fois,
- `x.to(device)`, `y.to(device)` pour chaque lot.

In [24]:
import time

def train_on(device, num_epochs=3):
    torch.manual_seed(123)
    model = NeuralNetwork(2, 2).to(device)                  # modèle -> device
    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

    start = time.time()
    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)               # données -> device
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return time.time() - start

print(f"Entraînement sur {device} : {train_on(device):.4f} s")

Entraînement sur cpu : 0.0030 s


### Q3. Comparer avec un autre device
Si un GPU est disponible, on compare avec le CPU.

> Sur un modèle aussi minuscule, le GPU n'est **pas** plus rapide (le temps d'envoyer les données au GPU domine). Le GPU devient indispensable pour les gros modèles comme GPT-2.

In [25]:
print(f"CPU : {train_on(torch.device('cpu')):.4f} s")
if device.type != "cpu":
    print(f"{device} : {train_on(device):.4f} s")
else:
    print("Pas de GPU disponible sur cette machine : comparaison impossible.")

CPU : 0.0040 s
Pas de GPU disponible sur cette machine : comparaison impossible.


---
## 🎯 Résumé de l'essentiel

| Notion | À retenir |
|---|---|
| Tenseur | tableau de nombres ; `.shape`, `.ndim`, `@` pour le produit matriciel |
| `view` / `reshape` | changent la forme sans changer les données ; `reshape` marche toujours |
| `requires_grad=True` | PyTorch suit les calculs pour pouvoir dériver |
| `loss.backward()` | calcule les gradients et les range dans `.grad` |
| `nn.Module` | `__init__` (couches) + `forward` (calcul) |
| `Dataset` / `DataLoader` | un exemple à la fois / des lots mélangés |
| Boucle d'entraînement | prédire → loss → `zero_grad` → `backward` → `step` |
| `torch.no_grad()` + `model.eval()` | pour prédire sans entraîner |
| Device | modèle **et** données sur le même device (`.to(device)`) |